In [1]:
import sys
if '../utils' not in sys.path: sys.path.append('../utils')

In [2]:
import json
import re
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


from few_shot_trials import TrialsGenerator, EvaluationDatasetBuilder

In [3]:
%run ../utils/results_explore.py

<Figure size 640x480 with 0 Axes>

In [4]:
def get_metadata_from_filename_or_log(filename, conversations):
    """Extracts explicit graph topology and configurations directly from string matches."""
    pattern = r"(LS|MTO|OTM)_M(\d+)_C(\d+)"
    match = re.search(pattern, filename)
    
    if match:
        return {
            "train_structure": match.group(1),
            "members_n": int(match.group(2)),
            "classes_n": int(match.group(3)),
        }
    
    if conversations:
        first_id = conversations[0].get("id", "")
        match = re.search(pattern, first_id)
        if match:
            return {
                "train_structure": match.group(1),
                "members_n": int(match.group(2)),
                "classes_n": int(match.group(3)),
            }
            
    raise ValueError(f"CRITICAL: Failed to parse structural metadata parameters from: {filename}")


def extract_responses_and_telemetry(conversations):
    """Parses conversations to capture raw API metrics and option predictions

    aligned using explicit key index parameters.
    """
    extracted_data = []

    for convo in conversations:
        trial_id = convo.get("id", "")
        if not trial_id.startswith("se_trial_"):
            continue

        idx_match = re.search(r"se_trial_(\d+)", trial_id)
        if not idx_match:
            continue
        trial_index = int(idx_match.group(1))

        metrics_block = convo.get("metrics", {})
        requests = convo.get("requests", [])

        is_empty_or_failed = True
        predicted_option = "NA"
        assistant_text = None

        if requests:
            contents = requests[0].get("contents", [])
            for item in contents:
                if item.get("role") == "CONTENT_ROLE_ASSISTANT":
                    parts = item.get("parts", [])
                    if parts and "text" in parts[0]:
                        assistant_text = parts[0]["text"]
                        break

            if assistant_text is not None and str(assistant_text).strip() != "":
                is_empty_or_failed = False
                matches = re.findall(r"O_[123]", str(assistant_text).upper())
                if matches:
                    predicted_option = matches[-1]
                else:
                    predicted_option = str(assistant_text).strip().upper()

        if is_empty_or_failed or not metrics_block:
            trial_metrics = {
                "trial_index": trial_index,
                "status": "No Response",
                "model_response": "NA",
                "is_silent_failure": 1,
                "inputTokens": 0,
                "outputTokens": 0,
                "inputTokensCostNanodollars": 0.0,
                "outputTokensCostNanodollars": 0.0,
                "totalBackendLatencyMs": np.nan,
                "throughput_tok_sec": np.nan,
            }
        else:
            in_tokens = int(metrics_block.get("inputTokens", 0))
            out_tokens = int(metrics_block.get("outputTokens", 0))
            latency_ms = float(metrics_block.get("totalBackendLatencyMs", 0))

            in_cost_nano = float(metrics_block.get("inputTokensCostNanodollars", 0))
            out_cost_nano = float(metrics_block.get("outputTokensCostNanodollars", 0))
            
            latency_sec = latency_ms / 1000.0
            throughput = (out_tokens / latency_sec) if latency_sec > 0 else np.nan

            trial_metrics = {
                "trial_index": trial_index,
                "status": "Model Response",
                "model_response": predicted_option,
                "is_silent_failure": 0,
                "inputTokens": in_tokens,
                "outputTokens": out_tokens,
                "inputTokensCostNanodollars": in_cost_nano,
                "outputTokensCostNanodollars": out_cost_nano,
                "totalBackendLatencyMs": latency_ms,
                "throughput_tok_sec": throughput,
            }

        extracted_data.append(trial_metrics)

    return pd.DataFrame(extracted_data)


def get_scores_graph_data(model_responses_df):
    """Maps predicted tokens to target answers and extracts clean string representations of nodes."""
    agent_info_df = model_responses_df.copy()
    
    agent_info_df["sample_member"] = agent_info_df["st_sample"].str[0]
    agent_info_df["comparison_member"] = agent_info_df["st_comparison"].str[0]
    
    agent_info_df["response_score"] = (
        agent_info_df["option_answer"] == agent_info_df["model_response"]
    ).astype(int)

    return agent_info_df

In [5]:
class AGIBenchmarkEngine:
    """Consolidates cross-trial model metrics into macro global,

    mid-tier capability property, and micro-level pair-wise node profiles
    with rigorous tracking of total attempted vs. valid non-NA trials.
    """
    def __init__(self, scored_trials_df: pd.DataFrame):
        self.df = scored_trials_df.copy()
        self._preprocess_data()

    def _preprocess_data(self):
        """Enforces structural types and systematically masks metrics on silent failures

        to enable accurate non-NA counting.
        """
        self.df["response_score"] = self.df["response_score"].astype(float)
        self.df["is_silent_failure"] = self.df["is_silent_failure"].astype(int)

        # CRITICAL MASKING: Explicitly convert failed telemetry to NaN.
        # This allows Pandas .count() to automatically exclude them.
        fail_mask = self.df["is_silent_failure"] == 1
        self.df.loc[fail_mask, "totalBackendLatencyMs"] = np.nan
        self.df.loc[fail_mask, "throughput_tok_sec"] = np.nan

    def generate_global_benchmark(self) -> pd.DataFrame:
        """Tier-1: Macro evaluation profiling tracking complete vs attempted allocations."""
        return (
            self.df.groupby(["model", "train_structure", "members_n", "classes_n"])
            .agg(
                macro_accuracy=("response_score", "mean"),
                silent_error_rate=("is_silent_failure", "mean"),
                total_attempted_trials=("model", "count"),
                valid_completed_trials=("totalBackendLatencyMs", "count"),
                median_totalBackendLatencyMs=("totalBackendLatencyMs", "median"),
                median_throughput_tok_sec=("throughput_tok_sec", "median"),
                median_inputTokens=("inputTokens", "median"),
                median_outputTokens=("outputTokens", "median"),
                median_inputTokensCostNanodollars=("inputTokensCostNanodollars", "median"),
                median_outputTokensCostNanodollars=("outputTokensCostNanodollars", "median"),
            )
            .reset_index()
        )

    def generate_property_benchmark(self) -> pd.DataFrame:
        """Tier-2: Cognitive property profiles evaluated via target non-NA entries."""
        return (
            self.df.groupby(["model", "train_structure", "sample_subset"])
            .agg(
                property_accuracy=("response_score", "mean"),
                silent_error_rate=("is_silent_failure", "mean"),
                total_attempted_trials=("model", "count"),
                valid_completed_trials=("totalBackendLatencyMs", "count"),
                median_totalBackendLatencyMs=("totalBackendLatencyMs", "median"),
                median_throughput_tok_sec=("throughput_tok_sec", "median"),
                median_inputTokens=("inputTokens", "median"),
                median_outputTokens=("outputTokens", "median"),
                median_inputTokensCostNanodollars=("inputTokensCostNanodollars", "median"),
                median_outputTokensCostNanodollars=("outputTokensCostNanodollars", "median"),
            )
            .reset_index()
        )

    def generate_pairwise_benchmark(self) -> pd.DataFrame:
        """Tier-3: Micro-transition maps isolating spatial constraints per edge mapping."""
        pairwise_perf = (
            self.df.groupby(["model", "train_structure", "sample_subset", "sample_member", "comparison_member"])
            .agg(
                pair_accuracy=("response_score", "mean"),
                silent_error_rate=("is_silent_failure", "mean"),
                total_attempted_trials=("model", "count"),
                valid_completed_trials=("totalBackendLatencyMs", "count"),
                median_totalBackendLatencyMs=("totalBackendLatencyMs", "median"),
                median_throughput_tok_sec=("throughput_tok_sec", "median"),
                median_inputTokens=("inputTokens", "median"),
                median_outputTokens=("outputTokens", "median"),
                median_inputTokensCostNanodollars=("inputTokensCostNanodollars", "median"),
                median_outputTokensCostNanodollars=("outputTokensCostNanodollars", "median"),
            )
            .reset_index()
        )

        # Enforce exact conditional constraint for stimulus equivalence chains
        pairwise_perf["node_hop_distance"] = pairwise_perf.apply(self._compute_ls_distance, axis=1)
        return pairwise_perf

    @staticmethod
    def _compute_ls_distance(row):
        """Calculates exact stimulus distance strings for LS topologies exclusively."""
        if str(row["train_structure"]).upper() != "LS":
            return np.nan
        
        try:
            pos_sample = ord(str(row["sample_member"]).upper())
            pos_comparison = ord(str(row["comparison_member"]).upper())
            return abs(pos_comparison - pos_sample)
        except Exception:
            return np.nan

In [6]:
models_files = os.listdir("bench_outputs")
print(len(models_files))
models_files

24


['SEB_Task1_LS_M4_C3-run_id_Run_1_anthropic_claude-opus-4-8default.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_anthropic_claude-sonnet-4-6default.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_deepseek-ai_deepseek-v3.2.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_google_gemini-3.1-pro-preview.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_openai_gpt-5.4-2026-03-05.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_openai_gpt-5.5-2026-04-23.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_qwen_qwen3-next-80b-a3b-instruct.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_xai_grok-4.20-0309-reasoning.run.json',
 'SEB_Task1_LS_M4_C3-run_id_Run_1_zai_glm-5.run.json',
 'SEB_Task2_MTO_M4_C3-run_id_Run_1_anthropic_claude-opus-4-8default.run.json',
 'SEB_Task2_MTO_M4_C3-run_id_Run_1_anthropic_claude-sonnet-4-6default.run.json',
 'SEB_Task2_MTO_M4_C3-run_id_Run_1_google_gemini-3.1-pro-preview.run.json',
 'SEB_Task2_MTO_M4_C3-run_id_Run_1_openai_gpt-5.4-2026-03-05.run.json',
 'SEB_Task2_MTO_M4_C3-run_id_Run_1_open

In [7]:
input_directory = "bench_outputs/"  # Update this folder path inside your Kaggle runtime

In [8]:
master_longform_buffer = []

In [9]:
for file_name in models_files:
    file_path = os.path.join(input_directory, file_name)
    
    if not os.path.exists(file_path):
        print(f"[WARNING] File not found in workspace: {file_path}. Skipping.")
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        json_data = json.load(f)

    conversations = json_data.get("conversations", [])
    
    # Isolate target tracking labels
    model_slug = json_data.get("modelVersion", {}).get("slug", "unknown").split("@")[0]
    if model_slug == "unknown":
        model_slug = file_name.split("run_id_Run_1_")[-1].replace(".run.json", "")

    # Extract meta parameters and base outputs
    run_metadata = get_metadata_from_filename_or_log(file_name, conversations)
    out_trials = extract_responses_and_telemetry(conversations)

    # Build testing environments via local environment modules
    generator = TrialsGenerator(**run_metadata)
    builder = EvaluationDatasetBuilder(generator)
    model_responses_df = builder.build_evaluation_dataset()
    
    model_responses_df.drop(
        columns=["few_shot_context", "formatted_test_trial"], 
        inplace=True, 
        errors="ignore"
    )

    # Map nodes back to structural keys
    reverse_translation_map = {v: k for k, v in generator.translation_map.items()}
    for col in ["st_sample", "st_comp1", "st_comp2", "st_comp3", "st_comparison"]:
        if col in model_responses_df.columns:
            model_responses_df[col] = model_responses_df[col].map(reverse_translation_map).fillna(model_responses_df[col])

    # Core position index alignment join step
    model_responses_df = model_responses_df.reset_index().rename(columns={"index": "trial_index"})
    merged_trial_df = pd.merge(model_responses_df, out_trials, on="trial_index", how="inner")

    # Run vectorized scoring calculations
    scored_df = get_scores_graph_data(merged_trial_df)

    # Broadcast structural properties as uniform series variables
    scored_df["model"] = model_slug
    scored_df["train_structure"] = run_metadata["train_structure"]
    scored_df["members_n"] = run_metadata["members_n"]
    scored_df["classes_n"] = run_metadata["classes_n"]

    master_longform_buffer.append(scored_df)

In [10]:
if master_longform_buffer:
    master_scored_df = pd.concat(master_longform_buffer, ignore_index=True)
    print(f"\nUnified evaluation array compiled. Base dimensions: {master_scored_df.shape}")

    # Initialize analysis calculations
    analysis_engine = AGIBenchmarkEngine(master_scored_df)
    
    df_tier1_global = analysis_engine.generate_global_benchmark()
    df_tier2_property = analysis_engine.generate_property_benchmark()
    df_tier3_pairwise = analysis_engine.generate_pairwise_benchmark()

    print("\n>>> DATA ANALYSIS EXTRACTION SUMMARY COMPLETE <<<")
    print(f"Tier 1 (Global Split):   {df_tier1_global.shape[0]} structural profile items.")
    print(f"Tier 2 (Property Split): {df_tier2_property.shape[0]} structural profile items.")
    print(f"Tier 3 (Pairwise Split): {df_tier3_pairwise.shape[0]} structural profile items.")

else:
    print("\n[CRITICAL ERROR] Buffer empty. Verify your data directory input settings paths.")


Unified evaluation array compiled. Base dimensions: (5616, 24)

>>> DATA ANALYSIS EXTRACTION SUMMARY COMPLETE <<<
Tier 1 (Global Split):   24 structural profile items.
Tier 2 (Property Split): 72 structural profile items.
Tier 3 (Pairwise Split): 312 structural profile items.


In [16]:
df_tier1_global

,model,train_structure,members_n,classes_n,macro_accuracy,silent_error_rate,total_attempted_trials,valid_completed_trials,median_totalBackendLatencyMs,median_throughput_tok_sec,median_inputTokens,median_outputTokens,median_inputTokensCostNanodollars,median_outputTokensCostNanodollars
0,anthropic/claude-opus-4-8,LS,4,3,0.658,0.000,234,234,"4,283.000",76.114,882.000,320.000,"4,410,000.000","8,000,000.000"
1,anthropic/claude-opus-4-8,MTO,4,3,0.953,0.000,234,234,"3,171.000",86.192,884.000,266.500,"4,420,000.000","6,662,500.000"
2,anthropic/claude-opus-4-8,OTM,4,3,0.944,0.000,234,234,"3,786.000",65.970,889.000,245.000,"4,445,000.000","6,125,000.000"
3,anthropic/claude-sonnet-4-6,LS,4,3,0.628,0.000,234,234,"8,143.500",62.555,709.000,493.500,"2,127,000.000","7,402,500.000"
4,anthropic/claude-sonnet-4-6,MTO,4,3,0.850,0.073,234,217,"8,089.000",67.277,697.000,460.500,"2,091,000.000","6,907,500.000"
5,anthropic/claude-sonnet-4-6,OTM,4,3,0.705,0.107,234,209,"6,659.000",57.780,698.000,349.500,"2,094,000.000","5,242,500.000"
6,deepseek-ai/deepseek-v3.2,LS,4,3,0.397,0.000,234,234,"1,464.000",2.876,608.000,4.000,"340,480.000","6,720.000"
7,google/gemini-3.1-pro-preview,LS,4,3,1.000,0.000,234,234,"7,126.500",120.352,615.000,862.000,"1,230,000.000","10,344,000.000"
8,google/gemini-3.1-pro-preview,MTO,4,3,1.000,0.000,234,234,"5,435.500",0.552,602.000,3.000,"1,204,000.000","6,690,000.000"
9,google/gemini-3.1-pro-preview,OTM,4,3,1.000,0.000,234,234,"7,032.500",0.427,612.000,3.000,"1,224,000.000","9,678,000.000"


In [12]:
df_tier2_property

,model,train_structure,sample_subset,property_accuracy,silent_error_rate,total_attempted_trials,valid_completed_trials,median_totalBackendLatencyMs,median_throughput_tok_sec,median_inputTokens,median_outputTokens,median_inputTokensCostNanodollars,median_outputTokensCostNanodollars
0,anthropic/claude-opus-4-8,LS,reflexivity,0.194,0.000,72,72,"4,711.000",76.114,882.000,342.500,"4,410,000.000","8,562,500.000"
1,anthropic/claude-opus-4-8,LS,symmetry,0.852,0.000,54,54,"3,743.000",72.700,881.000,279.500,"4,405,000.000","6,987,500.000"
2,anthropic/claude-opus-4-8,LS,transitivity,0.870,0.000,108,108,"4,105.500",76.880,882.000,316.000,"4,410,000.000","7,900,000.000"
3,anthropic/claude-opus-4-8,MTO,reflexivity,0.931,0.000,72,72,"3,356.500",82.190,884.000,274.500,"4,420,000.000","6,862,500.000"
4,anthropic/claude-opus-4-8,MTO,symmetry,0.963,0.000,54,54,"2,764.000",86.890,884.000,240.500,"4,420,000.000","6,012,500.000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,zai/glm-5,MTO,symmetry,0.944,0.019,54,53,"54,108.000",79.568,608.000,"4,291.500","608,000.000","13,732,800.000"
68,zai/glm-5,MTO,transitivity,0.444,0.519,108,52,"53,185.000",89.556,0.000,0.000,0.000,0.000
69,zai/glm-5,OTM,reflexivity,0.875,0.000,72,72,"109,667.500",70.905,607.000,"8,006.000","607,000.000","25,619,200.000"
70,zai/glm-5,OTM,symmetry,1.000,0.000,54,54,"43,427.500",74.873,607.000,"3,246.500","607,000.000","10,388,800.000"


In [13]:
df_tier3_pairwise

,model,train_structure,sample_subset,sample_member,comparison_member,pair_accuracy,silent_error_rate,total_attempted_trials,valid_completed_trials,median_totalBackendLatencyMs,median_throughput_tok_sec,median_inputTokens,median_outputTokens,median_inputTokensCostNanodollars,median_outputTokensCostNanodollars,node_hop_distance
0,anthropic/claude-opus-4-8,LS,reflexivity,A,A,0.333,0.000,18,18,"5,090.500",80.099,882.000,376.500,"4,410,000.000","9,412,500.000",0.000
1,anthropic/claude-opus-4-8,LS,reflexivity,B,B,0.222,0.000,18,18,"4,613.500",71.187,880.000,334.000,"4,400,000.000","8,350,000.000",0.000
2,anthropic/claude-opus-4-8,LS,reflexivity,C,C,0.056,0.000,18,18,"4,157.500",78.367,882.000,313.500,"4,410,000.000","7,837,500.000",0.000
3,anthropic/claude-opus-4-8,LS,reflexivity,D,D,0.167,0.000,18,18,"4,782.500",76.007,883.000,336.500,"4,415,000.000","8,412,500.000",0.000
4,anthropic/claude-opus-4-8,LS,symmetry,B,A,1.000,0.000,18,18,"3,566.000",77.166,881.000,236.500,"4,405,000.000","5,912,500.000",1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,zai/glm-5,OTM,transitivity,B,D,0.944,0.000,18,18,"61,137.500",84.105,607.000,"5,040.000","607,000.000","16,128,000.000",NaN
308,zai/glm-5,OTM,transitivity,C,B,0.778,0.056,18,17,"74,084.000",89.683,607.000,"6,198.500","607,000.000","19,835,200.000",NaN
309,zai/glm-5,OTM,transitivity,C,D,1.000,0.000,18,18,"42,935.500",103.604,607.000,"4,104.000","607,000.000","13,132,800.000",NaN
310,zai/glm-5,OTM,transitivity,D,B,0.333,0.667,18,6,"77,275.500",84.439,0.000,0.000,0.000,0.000,NaN


In [14]:
df_tier1_global.to_csv("agi_benchmark_global.csv", encoding="utf8")
df_tier2_property.to_csv("agi_benchmark_property.csv", encoding="utf8")
df_tier3_pairwise.to_csv("agi_benchmark_pairwise.csv", encoding="utf8")